---
title: "Stability heatmaps for PGC split-replication CV"
author: "Saikat Banerjee"
format:
  html: default
date: "2026-04-30"
file-modified: "2026-05-18"
abstract: "Visualize split-replication stability metrics from the Clorinn cross-validation pipeline. The notebook reads per-radius JSON summaries, converts by-k metrics into k-by-nuclear-norm grids, and plots heatmaps for projection distance, cumulative singular-value energy, and largest principal angle."
---

In [1]:
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from pymir import mpl_stylesheet
from pymir import mpl_utils
mpl_stylesheet.banskt_presentation(splinecolor = 'black', dpi = 300, fontsize=14)

from matplotlib import colormaps as mpl_cmaps
import matplotlib.colors as mpl_colors
from mpl_toolkits.axes_grid1 import make_axes_locatable

## Paths and analysis settings

`data_root` should point to the Snakemake output directory for the split-replication CV run. The notebook expects:

- per-radius stability JSON files under `stability/`
- the aggregate stability metric CSV under `summary/`

`prefix` identifies the model/solver combination. For example, `pgd_fw_nnm` matches files such as `pgd_fw_nnm_r8192.json`.

In [2]:
data_root = "/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cv_split_replication"
# Model/solver prefix used by the Snakemake output filenames.
PREFIXES = ["pgd_afw_nnm_corr", "pgd_fw_nnm"]

stability_out_dir = f"{data_root}/stability"
summary_out = f"{data_root}/summary/{PREFIXES[0]}_stability_metrics.csv"

fig_dir = Path("figures/split_replication_cv")
fig_dir.mkdir(parents=True, exist_ok=True)

## Read aggregate summary table

This CSV is not required for the heatmaps below, but displaying it is a useful sanity check that the expected pipeline outputs were generated.

In [3]:
summary_df = pd.read_csv(summary_out)
summary_df.head()

,nucnorm,mean_dist_k1,mean_dist_k2,mean_dist_k3,mean_dist_k4,mean_dist_k5,mean_dist_k6,mean_dist_k7,mean_dist_k8,mean_dist_k9,...,se_energy_k22,se_energy_k23,se_energy_k24,se_energy_k25,se_energy_k26,se_energy_k27,se_energy_k28,se_energy_k29,se_energy_k30,grid
0,1.0,0.021770,0.203261,0.529161,0.600799,0.648400,0.660207,0.659052,0.660754,0.667229,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,coarse
1,2.0,0.022354,0.274409,0.577392,0.564718,0.588592,0.624654,0.654132,0.668207,0.659147,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,coarse
2,4.0,0.023633,0.266199,0.536371,0.635676,0.662308,0.665486,0.668800,0.676398,0.675788,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,coarse
3,8.0,0.026768,0.326992,0.566602,0.625170,0.634612,0.643133,0.661838,0.670966,0.670403,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,coarse
4,16.0,0.034373,0.054074,0.169477,0.484521,0.601823,0.635789,0.658699,0.674419,0.663070,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,coarse


## Helper functions

The stability JSON files are stored one file per radius. Each file contains a `by_k` list with metrics evaluated at several ranks. The helper functions below convert this nested JSON structure into rectangular matrices suitable for heatmaps.

In [4]:
def load_stability_jsons(stability_dir, prefix):
    stability_dir = Path(stability_dir)
    paths = sorted(stability_dir.glob(f"{prefix}_r*.json"))

    records = []
    for path in paths:
        with open(path) as fh:
            rec = json.load(fh)
        rec["_path"] = str(path)
        records.append(rec)

    return records


def build_by_k_metrics_grid(records, metric, *, duplicate="error"):
    rows = []

    for rec in records:
        nucnorm = int(round(float(rec["nucnorm_full"])))

        for entry in rec["by_k"]:
            value = entry[metric]
            rows.append({
                "nucnorm": nucnorm, 
                "k": int(entry["k"]), 
                metric: float(value),
            })

    long_df = pd.DataFrame(rows)

    dup_mask = long_df.duplicated(subset=["nucnorm", "k"], keep=False)
    # Look for duplicates. This can happen if coarse and fine grids
    # contain the same nucnorm.
    if dup_mask.any():
        keep = "first" if duplicate == "first" else "last"
        long_df = (
            long_df
            .sort_values(["nucnorm", "k"])
            .drop_duplicates(subset=["nucnorm", "k"], keep=keep)
        )
    grid_df = (
        long_df
        .pivot(index="k", columns="nucnorm", values=metric)
        .sort_index(axis=0)
        .sort_index(axis=1)
    )

    grid_df.index.name = "k"
    grid_df.columns.name = "nucnorm"

    return grid_df

## Load stability metrics as heatmap grids

The three metrics below summarize complementary aspects of split-replication stability:

- `mean_dist`: mean projection distance between the top-`k` trait subspaces from two SNP splits. Lower is more stable.
- `mean_energy`: mean cumulative singular-value energy captured by the top-`k` directions. Higher means the selected `k` captures more fitted signal.
- `mean_gap_angle`: mean largest principal angle between split-specific top-`k` subspaces. Lower angles indicate more similar subspaces.

For plotting, duplicate radii are resolved with `duplicate="first"` because the coarse and fine radius grids can overlap.

In [5]:
records      = load_stability_jsons(stability_out_dir, PREFIXES[0])
dist_df      = build_by_k_metrics_grid(records, "mean_dist", duplicate="first")
energy_df    = build_by_k_metrics_grid(records, "mean_energy", duplicate="first")
gap_angle_df = build_by_k_metrics_grid(records, "mean_gap_angle", duplicate="first")

## Sanity check: projection-distance grid

Inspect the main metric before plotting. Rows are ranks `k`; columns are nuclear-norm radii `r`.

In [6]:
dist_df

nucnorm,1,2,4,8,16,32,64,128,256,512,...,4096,8192,9742,11585,13777,16384,19484,23170,27554,32768
k,,,,,,,,,,,,,,,,,,,,,
1,0.021770,0.022354,0.023633,0.026768,0.034373,0.060326,0.279794,0.104825,0.152003,0.316756,...,0.320522,0.506330,0.501743,0.396054,0.301842,0.215812,0.144993,0.116290,0.095999,0.079928
2,0.203261,0.274409,0.266199,0.326992,0.054074,0.295683,0.086641,0.081347,0.200903,0.353936,...,0.275586,0.227962,0.246326,0.252714,0.184844,0.165869,0.145464,0.125931,0.106317,0.102684
3,0.529161,0.577392,0.536371,0.566602,0.169477,0.096735,0.076713,0.111989,0.217589,0.190335,...,0.244471,0.343232,0.201930,0.178022,0.227835,0.247659,0.207810,0.179524,0.164217,0.137322
4,0.600799,0.564718,0.635676,0.625170,0.484521,0.072289,0.145789,0.088911,0.147440,0.208327,...,0.198169,0.247239,0.255817,0.262621,0.186821,0.171281,0.175579,0.275854,0.315842,0.177260
5,0.648400,0.588592,0.662308,0.634612,0.601823,0.248111,0.067116,0.083585,0.279528,0.250431,...,0.291562,0.211142,0.248516,0.253508,0.181319,0.158097,0.142910,0.145071,0.145925,0.149781
6,0.660207,0.624654,0.665486,0.643133,0.635789,0.352287,0.171369,0.086962,0.181595,0.190666,...,0.165505,0.209361,0.387707,0.216801,0.186497,0.184637,0.408319,0.194301,0.135193,0.120031
7,0.659052,0.654132,0.668800,0.661838,0.658699,0.410838,0.248090,0.079737,0.174569,0.226141,...,0.258335,0.298405,0.213943,0.196292,0.276207,0.281091,0.200430,0.224400,0.216707,0.178804
8,0.660754,0.668207,0.676398,0.670966,0.674419,0.477241,0.395293,0.127913,0.226393,0.209724,...,0.155959,0.226286,0.180093,0.173730,0.154254,0.290103,0.165114,0.123460,0.110062,0.102487
9,0.667229,0.659147,0.675788,0.670403,0.663070,0.492301,0.325064,0.236133,0.119586,0.216279,...,0.167974,0.140022,0.133098,0.141958,0.232874,0.125818,0.124066,0.126144,0.130920,0.128427


## Main cross-validation plot

Use this figure to decide where the projection distance stops improving materially as the nuclear-norm radius increases. The left panel gives the full `k` by `r` grid; the right panel shows one rank slice through the same grid.

In [13]:
def plot_heatmap(ax, X, rank_list, k_list,
        vmin = 0, vcenter = 0.9, vmax = 1.0):
    cmap1 = mpl_cmaps.get_cmap("YlOrRd").copy()
    cmap1.set_bad("w")
    norm1 = mpl_colors.TwoSlopeNorm(vmin = vmin, vcenter = vcenter, vmax = vmax)
    im1 = ax.imshow(X, cmap = cmap1, norm = norm1, origin = 'lower')
    
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.2)
    cbar = plt.colorbar(im1, cax=cax, fraction = 0.1)

    ax.set_xlabel("Nuclear norm constraint radius r")
    ax.set_ylabel("Rank k")
    ax.set_yticks(np.arange(len(k_list)))
    ax.set_yticklabels([str(int(k)) for k in k_list])
    ax.set_xticks(np.arange(len(rank_list)))
    ax.set_xticklabels([str(int(r)) for r in rank_list], rotation=90)    
    return

def make_projection_distance_figure(prefix,
        k_choose=24, vmin=0.09, vcenter=0.12, vmax=0.2):
    records = load_stability_jsons(stability_out_dir, prefix)
    dist_df = build_by_k_metrics_grid(records, "mean_dist", duplicate="first")
    rank_list = dist_df.columns.to_numpy()
    k_list = dist_df.index.to_numpy()
 
    fig = plt.figure(figsize=(15, 9), constrained_layout=True)
    gs = fig.add_gridspec(nrows=1, ncols=2, 
                          width_ratios=[1, 0.8],  # heatmap, line plot
                          wspace=0.2,
                         )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    plot_heatmap(ax1, dist_df.to_numpy(), rank_list, k_list, 
                 vmin=vmin, vcenter=vcenter, vmax=vmax)
    ax1.set_title("Mean projection distance across splits", pad = 20)

    # Plot a single row of the heatmap as a line plot. 
    # Makes it easier to see the plateau onset for one chosen rank.
    dist_vals = dist_df.loc[k_choose].to_numpy()
    ax2.plot(np.arange(len(dist_vals)), dist_vals, 'o-')
    ax2.set_xlabel("Nuclear norm constraint radius r")
    ax2.set_ylabel(f"Mean projection distance across splits at k={k_choose:d}")
    ax2.set_xticks(np.arange(len(dist_vals)))
    ax2.set_xticklabels(rank_list, rotation=90)
    ax2.set_title(f"Fixed-rank slice at k={k_choose}", pad = 20)
    
    fig.savefig(
        fig_dir / f"{prefix}_projection_distance.png",
        bbox_inches="tight",
    )
    plt.close(fig)
    # plt.show()
    

for prefix in PREFIXES:
    k_choose = 24
    vmin, vcenter, vmax = 0.09, 0.12, 0.20
    if prefix == "pgd_afw_nnm_corr":
        k_choose = 20
        vmin, vcenter, vmax = 0.1, 0.2, 0.4
    make_projection_distance_figure(prefix,
        k_choose=k_choose, vmin=vmin, vcenter=vcenter, vmax=vmax)

::: {.panel-tabset}

## pgd_fw_nnm

![](figures/split_replication_cv/pgd_fw_nnm_projection_distance.png)

## pgd_afw_nnm

![](figures/split_replication_cv/pgd_afw_nnm_corr_projection_distance.png)

:::

## Diagnostics

These plots are secondary checks. They help distinguish a real stability plateau from artifacts caused by changing effective rank or unstable high-dimensional directions.

In [15]:
def make_diagnostics_energy_angle_figure(prefix):
    records      = load_stability_jsons(stability_out_dir, prefix)
    energy_df    = build_by_k_metrics_grid(records, "mean_energy", duplicate="first")
    gap_angle_df = build_by_k_metrics_grid(records, "mean_gap_angle", duplicate="first")
    rank_list = energy_df.columns.to_numpy()
    k_list = energy_df.index.to_numpy()

    fig = plt.figure(figsize=(15, 9), constrained_layout=True)
    gs = fig.add_gridspec(nrows=1, ncols=2, 
                          width_ratios=[1, 1],
                          wspace=0.2,
                         )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    plot_heatmap(ax1, energy_df.to_numpy(), rank_list, k_list, vmin = 0.8, vcenter = 0.9, vmax = 1.0)
    ax1.set_title("Mean cumulative singular-value energy", pad = 20)

    rank_list = gap_angle_df.columns.to_numpy()
    k_list = gap_angle_df.index.to_numpy()
    plot_heatmap(ax2, gap_angle_df.to_numpy(), rank_list, k_list, vmin = 0, vcenter = 45, vmax = 90.0)
    ax2.set_title("Mean largest principal angle across splits", pad = 20)
    
    fig.savefig(
        fig_dir / f"{prefix}_energy_gap_angle.png",
        bbox_inches="tight",
    )
    plt.close(fig)
    
for prefix in PREFIXES:
    make_diagnostics_energy_angle_figure(prefix)

::: {.panel-tabset}

## pgd_fw_nnm

![](figures/split_replication_cv/pgd_fw_nnm_energy_gap_angle.png)

## pgd_afw_nnm

![](figures/split_replication_cv/pgd_afw_nnm_corr_energy_gap_angle.png)

:::